In [1]:
from calidad_pqrs.utils import load_directory, load_data, mapping_data, define_service, define_f3, clean_text_TfIdf, load_model, build_predictions_dataframe, create_probability_col, format_results, load_thresholds, export_results
from calidad_pqrs.models.predict import process_validation, causes_validation, final_validation
from calidad_pqrs.config import MODEL_PROCESS_DIR, MODEL_CAUSES_DIR

In [2]:
paths = load_directory(directory='Predict')

In [3]:
complaints = load_data(paths)

Leyendo las descripciones de las quejas...


In [4]:
complaints['RAC_process_raw'] = complaints['Proceso'].copy()
complaints['RAC_causes_raw'] = complaints['Causa'].copy()

In [5]:
complaints = mapping_data(complaints)

Homologando procesos y causas con poca participación...


In [6]:
complaints['Filtro 4_map'] = complaints.apply(define_service, axis=1)
complaints['Filtro 3_map'] = complaints['Filtro 3'].apply(define_f3)

In [7]:
complaints = clean_text_TfIdf(complaints)

Removiendo categorías gramaticales que aportan poca información...
Lematizando palabras y removiendo stopwords...


In [8]:
# Estas columnas son necesarias, ya que la comparación la hago contra los datos de entrenamiento y estos tuvieron homologaciones.
# El sentido está en la función mapping_data, donde se hace la homologación de los procesos y causas.

complaints.rename(
    columns={
        'Proceso': 'Temp_RAC_Process',
        'Causa': 'Temp_RAC_Causes'
    }, inplace=True
)

In [11]:
process_model = load_model(MODEL_PROCESS_DIR, 'salud_process_classifier.pkl')

In [12]:
probability_matrix = build_predictions_dataframe(
    model = process_model,
    X_test = complaints['Descripción_TfIdf'],
    true_class_col = 'RAC_Process',
    y_test= complaints['Temp_RAC_Process'],
    predicted_class_col = 'Proceso',
    dataset_original = complaints,
    drop_cols = ['Temp_RAC_Process', 'grammatical_categories_removed', 'stopwords_removed', 'Filtro 3', 'Filtro 4']
)

In [13]:
probability_matrix = create_probability_col(
    dataset = probability_matrix,
    col_name = 'Process_Probability'
    )

In [14]:
causes_model = load_model(MODEL_CAUSES_DIR, 'salud_causes_classifier.pkl')

In [15]:
results = build_predictions_dataframe(
    model = causes_model,
    X_test = probability_matrix[['Descripción_TfIdf', 'Proceso', 'Filtro 4_map', 'Filtro 3_map']],
    true_class_col = 'RAC_Causes',
    y_test= probability_matrix['Temp_RAC_Causes'],
    predicted_class_col = 'Causa_Sugerida',
    dataset_original = probability_matrix,
    drop_cols = ['Descripción_TfIdf', 'Filtro 4_map', 'Filtro 3_map']
)

In [16]:
results = create_probability_col(
    dataset = results,
    col_name = 'Causes_Probability'
    )

In [17]:
results.rename(columns={'Proceso': 'Proceso_Sugerido'}, inplace=True)

In [18]:
process_thresholds = load_thresholds(filename = 'process_thresholds.parquet')
causes_thresholds = load_thresholds(filename = 'causes_thresholds.parquet')

In [19]:
results['Validated_Process_Label'] = results.apply(process_validation, axis=1)

results['Validated_Causes_Label'] = results.apply(causes_validation, axis=1)

results['Final Validation'] = results.apply(final_validation, axis=1)

In [18]:
results = format_results(results)

In [20]:
export_results(results, 'Alertas calidad.xlsx')